# Knowledge that Accumulates: Multi-Quarter Rollout

The `ai2analytics.knowledge` module logs every pipeline run and lets you
synthesise patterns across runs. We simulate **four quarterly rollouts** of
segmentation pipelines (US Q3, EU Q3, US Q4, BRIC Q4) into a JSONL-backed
knowledge store, then show how the retriever surfaces what would otherwise be
tribal knowledge.

In [1]:
# Run once per session. In Colab, this installs into the runtime;
# in local Jupyter, it goes to the active kernel's environment.
import sys, subprocess
def _ensure(pkg, import_name=None):
    name = import_name or pkg.split('[')[0].split('==')[0]
    try:
        __import__(name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

# In Colab/Jupyter, install ai2analytics from GitHub. If it's already
# installed locally (editable), this is a no-op.
try:
    import ai2analytics  # noqa: F401
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'git+https://github.com/jamesyoung93/AI2Analytics.git'])

# Plot helpers (matplotlib is already a hard dep of ai2analytics)
_ensure('matplotlib')
print('Setup complete.')

Setup complete.


## 1. Spin up an in-memory knowledge store

In [2]:
import os, tempfile
from ai2analytics.knowledge import (
    DecisionStore, DecisionRecord,
    ContextStore, ContextEntry,
    KnowledgeRetriever,
)

tmpdir = tempfile.mkdtemp(prefix='knowledge_demo_')
decisions = DecisionStore(backend='json', path=os.path.join(tmpdir, 'decisions.jsonl'))
context   = ContextStore(backend='json', path=os.path.join(tmpdir, 'context.jsonl'))
print('Stores at:', tmpdir)

Stores at: C:\Users\Admin\AppData\Local\Temp\knowledge_demo_tcoteha3


## 2. Run four pipelines and log each one

In [3]:
import time
from ai2analytics.datasets import us_hcp, eu_account, bric
from ai2analytics.templates.segmentation import SegmentationConfig, SegmentationPipeline

run_log = []

def run_and_log(name, region, quarter, dfs_key, cfg, df, tags):
    out = SegmentationPipeline().run(cfg, dataframes={'entity_data': df})
    rid = decisions.log(DecisionRecord(
        template_name='segmentation',
        config_dict={
            'analysis_name': cfg.analysis_name,
            'col_entity_id': cfg.col_entity_id,
            'feature_columns': list(cfg.feature_columns),
            'method': cfg.method,
            'n_segments': cfg.n_segments,
        },
        data_profile=f'{region} {dfs_key} table, {len(df):,} rows',
        outcome_notes=f'Completed; {out.summary_stats["method"]} method.',
        outcome_metrics={
            'silhouette_score': float(out.summary_stats['silhouette_score']),
            'n_entities': out.summary_stats['n_entities'],
            'n_segments': out.summary_stats['n_segments'],
        },
        tags=tags,
    ))
    run_log.append((name, rid, out.summary_stats['silhouette_score']))
    time.sleep(0.05)
    return rid

# US Q3
us_q3 = us_hcp.generate_all(seed=42)
run_and_log('US Q3', 'US', 'Q3', 'hcp_reference',
    SegmentationConfig(analysis_name='us_hcp_q3', col_entity_id='npi',
        feature_columns=['IL_17_TRX_L12M', 'IL_23_TRX_L12M'],
        n_segments=4, method='kmeans', output_csv='_q3_us.csv'),
    us_q3['hcp_reference'], tags=['us', 'hcp', 'q3-2025'])

# EU Q3
eu_q3 = eu_account.generate_all(seed=43)
run_and_log('EU Q3', 'EU', 'Q3', 'account_reference',
    SegmentationConfig(analysis_name='eu_account_q3', col_entity_id='PRESCRIBER_ID',
        feature_columns=['UNITS_SOLD_L12M', 'TIER'],
        n_segments=3, method='auto', output_csv='_q3_eu.csv'),
    eu_q3['account_reference'], tags=['eu', 'account', 'q3-2025'])

# US Q4 (different seed = different data)
us_q4 = us_hcp.generate_all(seed=142)
run_and_log('US Q4', 'US', 'Q4', 'hcp_reference',
    SegmentationConfig(analysis_name='us_hcp_q4', col_entity_id='npi',
        feature_columns=['IL_17_TRX_L12M', 'IL_23_TRX_L12M'],
        n_segments=4, method='kmeans', output_csv='_q4_us.csv'),
    us_q4['hcp_reference'], tags=['us', 'hcp', 'q4-2025'])

# BRIC Q4
br_q4 = bric.generate_all(seed=44)
import pandas as pd
qsum = (br_q4['quarterly_performance'].groupby('ACCOUNT_ID', as_index=False)
        .agg(REVENUE_L12M=('REVENUE_LOCAL', 'sum'),
             AVG_PATIENT_STARTS=('PATIENT_STARTS', 'mean')))
br_view = br_q4['account_master'].merge(qsum, on='ACCOUNT_ID', how='left').fillna(0)
run_and_log('BRIC Q4', 'BRIC', 'Q4', 'account_master',
    SegmentationConfig(analysis_name='bric_account_q4', col_entity_id='ACCOUNT_ID',
        feature_columns=['REVENUE_L12M', 'AVG_PATIENT_STARTS'],
        n_segments=4, method='auto', output_csv='_q4_bric.csv'),
    br_view, tags=['bric', 'account', 'q4-2025'])

import pandas as pd
pd.DataFrame(run_log, columns=['run', 'run_id', 'silhouette']).round(3)

Generating synthetic US HCP data (seed=42) ...
  hcp_reference:        5,000 rows


  hcp_weekly:         114,011 rows


  calls:               32,726 rows
  team_a_alignment:     4,255 rows
  team_b_alignment:     3,964 rows
  portfolio_decile:     5,000 rows
  priority_targets:       500 rows
Done.


  PIPELINE: us_hcp_q3 Segmentation

STAGE 1: Loading data
  Entity data:  5,000 rows (in-memory)
  Feature cols: 2 specified
  Done.

STAGE 2: Preparing features
  Missing handling: filled with column median
  Feature matrix:   5,000 entities x 2 features
  Normalization:    standard
  Done.

STAGE 3: Fitting segments
  Method: kmeans


  Segments:     4
  Silhouette:   0.6142
  Done.

STAGE 4: Building output
  Assignments:  5,000 entities
  Profiles:     4 segments x 2 features
  Done.

STAGE 5: Writing output
  CSV:   _q3_us.csv (5,000 rows)
  Done.


PIPELINE COMPLETE
  Analysis:     us_hcp_q3
  Entities:     5,000
  Segments:     4
  Method:       kmeans
  Silhouette:   0.6142
    Segment 0: 315 entities
    Segment 1: 3,419 entities
    Segment 2: 917 entities
    Segment 3: 349 entities
  Output CSV:   _q3_us.csv
Generating synthetic EU account data (seed=43) ...


  account_reference:    3,000 rows


...

  Segments:     4
  Silhouette:   0.6088
  Done.

STAGE 4: Building output
  Assignments:  5,000 entities
  Profiles:     4 segments x 2 features
  Done.

STAGE 5: Writing output
  CSV:   _q4_us.csv (5,000 rows)
  Done.


PIPELINE COMPLETE
  Analysis:     us_hcp_q4
  Entities:     5,000
  Segments:     4
  Method:       kmeans
  Silhouette:   0.6088
    Segment 0: 3,377 entities
    Segment 1: 424 entities
    Segment 2: 893 entities
    Segment 3: 306 entities
  Output CSV:   _q4_us.csv
Generating synthetic BRIC data (seed=44) ...
  account_master:             2,000 rows


  quarterly_performance:     11,440 rows
  engagement:                11,440 rows
  sales_alignment:            1,624 rows
Done.


  PIPELINE: bric_account_q4 Segmentation

STAGE 1: Loading data
  Entity data:  2,000 rows (in-memory)
  Feature cols: 2 specified
  Done.

STAGE 2: Preparing features
  Missing handling: filled with column median
  Feature matrix:   2,000 entities x 2 features
  Normalization:    standard
  Done.

STAGE 3: Fitting segments
  Method: auto (trying both kmeans and hierarchical)


C:\Users\Admin\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=8.
  warnings.warn(


  KMeans       (k=4): silhouette=0.5651
  Hierarchical (k=4): silhouette=0.5526
  Selected: kmeans
  Segments:     4
  Silhouette:   0.5651
  Done.

STAGE 4: Building output
  Assignments:  2,000 entities
  Profiles:     4 segments x 2 features
  Done.

STAGE 5: Writing output
  CSV:   _q4_bric.csv (2,000 rows)
  Done.


PIPELINE COMPLETE
  Analysis:     bric_account_q4
  Entities:     2,000
  Segments:     4
  Method:       kmeans
  Silhouette:   0.5651
    Segment 0: 1,249 entities
    Segment 1: 220 entities
    Segment 2: 500 entities
    Segment 3: 31 entities
  Output CSV:   _q4_bric.csv


,run,run_id,silhouette
0,US Q3,517baf850874,0.614
1,EU Q3,84f8dabd3fbf,0.572
2,US Q4,c4955bed038c,0.609
3,BRIC Q4,92fbcd9f55b7,0.565


## 3. Curate context: synthesise patterns we noticed across the four runs

In production these come out of `ContextStore.extract_from_decisions(llm)` —
here we add a few by hand to show the shape.

In [4]:
from ai2analytics.knowledge import ContextEntry

context.add(ContextEntry(
    scope={'region': 'us'}, category='column_mapping',
    title='US HCP tables use "npi" (10-digit int) as entity ID',
    content='Always present; lowercase. Map col_entity_id="npi".',
    template_name='segmentation', confidence=0.95,
))
context.add(ContextEntry(
    scope={'region': 'eu'}, category='column_mapping',
    title='EU account tables use PRESCRIBER_ID (ACC-XXXXX)',
    content='String identifier. Map col_entity_id="PRESCRIBER_ID".',
    template_name='segmentation', confidence=0.92,
))
context.add(ContextEntry(
    scope={'region': 'bric'}, category='adapter_pattern',
    title='BRIC quarterly_performance must be aggregated to one row per ACCOUNT_ID',
    content='Group by ACCOUNT_ID and sum REVENUE_LOCAL / mean COMPLIANCE before clustering.',
    template_name='segmentation', confidence=0.85,
))
context.add(ContextEntry(
    scope={}, category='config_preference',
    title='method="auto" outperforms fixed kmeans on EU/BRIC',
    content='When you do not have a strong prior, let the pipeline choose between '
            'kmeans and hierarchical via silhouette.',
    template_name='segmentation', confidence=0.78,
))
print('Logged', len(context.query(limit=100)), 'context entries.')

Logged 4 context entries.


## 4. The retriever: what would the LLM see for a *new* US run?

This is the block the AI session prepends to its system prompt when a fresh
brand asks for segmentation. Past column mappings + learned patterns get
surfaced automatically.

In [5]:
retriever = KnowledgeRetriever(decision_store=decisions, context_store=context,
                               max_decisions=5, max_context=5)

print(retriever.retrieve_for_analysis(
    template_name='segmentation',
    scope={'region': 'us'},
))

--- ANALYSIS KNOWLEDGE ---
PAST COLUMN MAPPINGS:

  Run 92fbcd9f55b7 (segmentation):
    Outcome: Completed; kmeans method.
  Run c4955bed038c (segmentation):
    Outcome: Completed; kmeans method.
  Run 84f8dabd3fbf (segmentation):
    Outcome: Completed; kmeans method.
  Run 517baf850874 (segmentation):
    Outcome: Completed; kmeans method.

MAPPING PATTERNS:

  [column_mapping] US HCP tables use "npi" (10-digit int) as entity ID (95%)
    Always present; lowercase. Map col_entity_id="npi".
--- END ANALYSIS KNOWLEDGE ---


## 5. The same retrieval scoped to a region the model has *less* experience with

Notice how the BRIC scope brings up the aggregation adapter pattern.

In [6]:
print(retriever.retrieve_for_analysis(
    template_name='segmentation',
    scope={'region': 'bric'},
))

--- ANALYSIS KNOWLEDGE ---
PAST COLUMN MAPPINGS:

  Run 92fbcd9f55b7 (segmentation):
    Outcome: Completed; kmeans method.
  Run c4955bed038c (segmentation):
    Outcome: Completed; kmeans method.
  Run 84f8dabd3fbf (segmentation):
    Outcome: Completed; kmeans method.
  Run 517baf850874 (segmentation):
    Outcome: Completed; kmeans method.
--- END ANALYSIS KNOWLEDGE ---


## 6. Inspect the raw store on disk

In [7]:
import json
from pathlib import Path

print('--- decisions.jsonl ---')
for line in Path(decisions.path).read_text().splitlines()[:6]:
    d = json.loads(line)
    print(f"  {d['run_id']}  {d['config_dict']['analysis_name']:<22s}  silhouette={d['outcome_metrics']['silhouette_score']:.3f}")

print('\n--- context.jsonl ---')
for line in Path(context.path).read_text().splitlines():
    d = json.loads(line)
    print(f"  [{d['category']:<18s}] {d['title']}  ({d['confidence']:.0%})")

--- decisions.jsonl ---
  517baf850874  us_hcp_q3               silhouette=0.614
  84f8dabd3fbf  eu_account_q3           silhouette=0.572
  c4955bed038c  us_hcp_q4               silhouette=0.609
  92fbcd9f55b7  bric_account_q4         silhouette=0.565

--- context.jsonl ---
  [column_mapping    ] US HCP tables use "npi" (10-digit int) as entity ID  (95%)
  [column_mapping    ] EU account tables use PRESCRIBER_ID (ACC-XXXXX)  (92%)
  [adapter_pattern   ] BRIC quarterly_performance must be aggregated to one row per ACCOUNT_ID  (85%)
  [config_preference ] method="auto" outperforms fixed kmeans on EU/BRIC  (78%)


## What this enables

- **Onboarding new analysts.** They inherit every prior config decision via
  `KnowledgeRetriever`, not via tribal knowledge.
- **Cross-team consistency.** Region-scoped scopes mean the US team's mapping
  conventions are surfaced to anyone running US data, even if they're new.
- **Cheaper LLM workflows.** Past auto-detected fields shorten the conversation
  the AI session needs to have on each new brand.

The decision/context schemas live in `ai2analytics.knowledge`. Both stores
support a Spark `delta` backend for production — flip `backend='delta'` and
point them at a Unity Catalog table.